# Pseudo EOF patterns based on GMM - Monthly Data (No DJF Selection)

In [ ]:
import os
from importlib import reload
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy as ctp
import seaborn as sns
from sklearn import mixture, decomposition
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import cartopy.mpl.ticker as cticker
import cartopy.feature as feature
import matplotlib.ticker as mticker
from datetime import datetime, timedelta
import xesmf as xe

In [ ]:
import preproc
import utils
import utenso
import eof
import utdata
import utstats
import metric
import geoplot as gpl
plt.style.use("/home/gsato/latgmm/paper.mplstyle")

In [ ]:
ds_pi = xr.open_dataset("/groups/XDU5/Go/research_data/latgmm/sst_pi_500.nc", decode_timedelta=True)
ds_mh = xr.open_dataset("/groups/XDU5/Go/research_data/latgmm/sst_mh_500.nc", decode_timedelta=True)

### 1. Interpolate data on a curvilinear grid to a rectilinear grid for PCA analysis

In [ ]:
def regrid(ds):
    
    dr = ds
    
    #1 rename the coordinate names to lon and lat
    ds = ds.rename({"TLONG": "lon", "TLAT": "lat"})

    #2 regrid to a global 1deg * 1deg grid
    ds_out = xr.Dataset(
    {
        "lat": (["lat"], np.arange(-90, 90, 1.0), {"units": "degrees_north"}),
        "lon": (["lon"], np.arange(-180, 180, 1.0), {"units": "degrees_east"}),
    }
    )

    #3 Regridding
    regridder = xe.Regridder(ds, ds_out, "bilinear")
    dr_out = regridder(dr)
    
    return dr_out

In [ ]:
temp_pi_reg = regrid(ds_pi)
temp_mh_reg = regrid(ds_mh)

In [ ]:
temp_pi_reg

In [ ]:
# Extract SST variable
sst_pi_500 = temp_pi_reg["sst"]
sst_mh_500 = temp_mh_reg["sst"]

In [ ]:
# Getting climatologies and anomalies (monthly climatology)
sst_pi_clim = sst_pi_500.groupby('time.month').mean("time")
sst_pi_anom = sst_pi_500.groupby('time.month')-sst_pi_clim

sst_mh_clim = sst_mh_500.groupby('time.month').mean("time")
sst_mh_anom = sst_mh_500.groupby('time.month')-sst_mh_clim

In [ ]:
# Check NaN values
print("PI NaN values in first month:", sst_pi_anom[0,:,:].isnull().sum().values)
print("MH NaN values in first month:", sst_mh_anom[0,:,:].isnull().sum().values)

In [ ]:
# Making new datasets as xarray Dataset
sst_pi_anom = xr.Dataset(
    {
        'ssta': (["time", 'lat', 'lon'], sst_pi_anom[:,:,:].values),
    },
    coords={"time": sst_pi_anom["time"],
        'lat': sst_pi_anom["lat"],
        'lon': sst_pi_anom["lon"],
    },
)

sst_mh_anom = xr.Dataset(
    {
        'ssta': (["time", 'lat', 'lon'], sst_mh_anom[:,:,:].values),
    },
    coords={"time": sst_pi_anom["time"],
        'lat': sst_pi_anom["lat"],
        'lon': sst_pi_anom["lon"],
    },
)

### 2. Perform EOF analysis on ALL MONTHLY data (no DJF selection)

In [ ]:
# Using all monthly data for EOF analysis
# Reshape data for EOF analysis: (time, space)
# Flatten the spatial dimensions

pi_data_2d = sst_pi_anom['ssta'].values.reshape(sst_pi_anom['ssta'].shape[0], -1)
mh_data_2d = sst_mh_anom['ssta'].values.reshape(sst_mh_anom['ssta'].shape[0], -1)

# Remove NaN rows
pi_valid_mask = ~np.isnan(pi_data_2d).any(axis=1)
mh_valid_mask = ~np.isnan(mh_data_2d).any(axis=1)

pi_data_clean = pi_data_2d[pi_valid_mask]
mh_data_clean = mh_data_2d[mh_valid_mask]

print(f"PI clean data shape: {pi_data_clean.shape}")
print(f"MH clean data shape: {mh_data_clean.shape}")

In [ ]:
# PCA on PI data using all months
pca_pi = decomposition.PCA(n_components=2)
pc_pi = pca_pi.fit_transform(pi_data_clean)

print(f"PI explained variance ratio: {pca_pi.explained_variance_ratio_}")
print(f"PI total explained variance: {pca_pi.explained_variance_ratio_.sum()}")

In [ ]:
# PCA on MH data using all months
pca_mh = decomposition.PCA(n_components=2)
pc_mh = pca_mh.fit_transform(mh_data_clean)

print(f"MH explained variance ratio: {pca_mh.explained_variance_ratio_}")
print(f"MH total explained variance: {pca_mh.explained_variance_ratio_.sum()}")

### 3. Visualize PC1-PC2 scatter for all monthly data

In [ ]:
# Create scatter plot for all monthly data
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PI scatter
axes[0].scatter(pc_pi[:, 0], pc_pi[:, 1], alpha=0.5, s=20)
axes[0].set_xlabel(f'PC1 ({pca_pi.explained_variance_ratio_[0]:.2%})')
axes[0].set_ylabel(f'PC2 ({pca_pi.explained_variance_ratio_[1]:.2%})')
axes[0].set_title('PI - All Monthly Data')
axes[0].grid(True, alpha=0.3)

# MH scatter
axes[1].scatter(pc_mh[:, 0], pc_mh[:, 1], alpha=0.5, s=20, color='orange')
axes[1].set_xlabel(f'PC1 ({pca_mh.explained_variance_ratio_[0]:.2%})')
axes[1].set_ylabel(f'PC2 ({pca_mh.explained_variance_ratio_[1]:.2%})')
axes[1].set_title('MH - All Monthly Data')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('PC1_PC2_all_months.png', dpi=150, bbox_inches='tight')
plt.show()

### 4. Gaussian Mixture Model clustering on all monthly PC1-PC2

In [ ]:
# GMM for PI data using all months
gmm_pi = mixture.GaussianMixture(n_components=3, random_state=42, n_init=10)
pi_labels = gmm_pi.fit_predict(pc_pi)

print(f"PI GMM - BIC: {gmm_pi.bic(pc_pi):.2f}")
print(f"PI GMM - AIC: {gmm_pi.aic(pc_pi):.2f}")
print(f"PI cluster sizes: {np.bincount(pi_labels)}")

In [ ]:
# GMM for MH data using all months
gmm_mh = mixture.GaussianMixture(n_components=3, random_state=42, n_init=10)
mh_labels = gmm_mh.fit_predict(pc_mh)

print(f"MH GMM - BIC: {gmm_mh.bic(pc_mh):.2f}")
print(f"MH GMM - AIC: {gmm_mh.aic(pc_mh):.2f}")
print(f"MH cluster sizes: {np.bincount(mh_labels)}")

### 5. Visualize GMM clustering results for all monthly data

In [ ]:
# Create scatter plot with GMM clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PI scatter with GMM clusters
scatter_pi = axes[0].scatter(pc_pi[:, 0], pc_pi[:, 1], c=pi_labels, cmap='viridis', alpha=0.6, s=30)
axes[0].set_xlabel(f'PC1 ({pca_pi.explained_variance_ratio_[0]:.2%})')
axes[0].set_ylabel(f'PC2 ({pca_pi.explained_variance_ratio_[1]:.2%})')
axes[0].set_title('PI - GMM Clustering (All Months)')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter_pi, ax=axes[0], label='Cluster')

# MH scatter with GMM clusters
scatter_mh = axes[1].scatter(pc_mh[:, 0], pc_mh[:, 1], c=mh_labels, cmap='viridis', alpha=0.6, s=30)
axes[1].set_xlabel(f'PC1 ({pca_mh.explained_variance_ratio_[0]:.2%})')
axes[1].set_ylabel(f'PC2 ({pca_mh.explained_variance_ratio_[1]:.2%})')
axes[1].set_title('MH - GMM Clustering (All Months)')
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter_mh, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.savefig('GMM_clusters_all_months.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot GMM covariance ellipses
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Helper function to plot covariance ellipses
def plot_gmm_ellipses(ax, gmm, pc_data, labels, title):
    colors = ['red', 'green', 'blue']
    ax.scatter(pc_data[:, 0], pc_data[:, 1], c=labels, cmap='viridis', alpha=0.4, s=20)
    
    # Plot GMM means
    ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='black', marker='x', s=200, linewidths=3)
    
    # Plot covariance ellipses
    from matplotlib.patches import Ellipse
    for i, (mean, cov) in enumerate(zip(gmm.means_, gmm.covariances_)):
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
        width, height = 2 * 2 * np.sqrt(eigenvalues)
        ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle,
                         facecolor='none', edgecolor=colors[i], linewidth=2, label=f'Cluster {i}')
        ax.add_patch(ellipse)
    
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plot_gmm_ellipses(axes[0], gmm_pi, pc_pi, pi_labels, 'PI - GMM Covariances (All Months)')
plot_gmm_ellipses(axes[1], gmm_mh, pc_mh, mh_labels, 'MH - GMM Covariances (All Months)')

axes[0].set_xlabel(f'PC1 ({pca_pi.explained_variance_ratio_[0]:.2%})')
axes[0].set_ylabel(f'PC2 ({pca_pi.explained_variance_ratio_[1]:.2%})')
axes[1].set_xlabel(f'PC1 ({pca_mh.explained_variance_ratio_[0]:.2%})')
axes[1].set_ylabel(f'PC2 ({pca_mh.explained_variance_ratio_[1]:.2%})')

plt.tight_layout()
plt.savefig('GMM_ellipses_all_months.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print("="*60)
print("MONTHLY GMM CLUSTERING ANALYSIS (NO DJF SELECTION)")
print("="*60)
print(f"\nPI Data:")
print(f"  Total samples: {len(pi_labels)}")
print(f"  PC1 explained variance: {pca_pi.explained_variance_ratio_[0]:.2%}")
print(f"  PC2 explained variance: {pca_pi.explained_variance_ratio_[1]:.2%}")
print(f"  GMM BIC: {gmm_pi.bic(pc_pi):.2f}")
print(f"  Cluster distribution: {dict(zip(*np.unique(pi_labels, return_counts=True)))}")

print(f"\nMH Data:")
print(f"  Total samples: {len(mh_labels)}")
print(f"  PC1 explained variance: {pca_mh.explained_variance_ratio_[0]:.2%}")
print(f"  PC2 explained variance: {pca_mh.explained_variance_ratio_[1]:.2%}")
print(f"  GMM BIC: {gmm_mh.bic(pc_mh):.2f}")
print(f"  Cluster distribution: {dict(zip(*np.unique(mh_labels, return_counts=True)))}")
print("="*60)